In [4]:
import json
import os
import time
import random
import statistics
import pandas as pd
from pathlib import Path
from google.genai import types

ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)

from lib.experiment_utils import create_client, load_best_attempts_df
from lib.llm_batch_analyzer import clean_json_response, format_submissions
from lib.prompts import build_v3_prompt
from utils.constants import PROBLEM_PROMPT_PATH

MODEL_ID = 'gemini-2.5-flash'
SLEEP_SECONDS = 7.0
OUTPUT_DIR = Path('results/learning_curve')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = create_client()
problem_prompts_df = pd.read_csv(PROBLEM_PROMPT_PATH)
best_attempts_df = load_best_attempts_df()

KC_COLUMNS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]

batch = pd.read_csv(ROOT / 'results' / '06_batch_30students' / 'batch_comparison_30students.csv')
STUDENT_IDS = sorted(batch['SubjectID'].unique().tolist())
student_clusters = batch.drop_duplicates('SubjectID')[['SubjectID', 'Cluster']].set_index('SubjectID')['Cluster'].to_dict()

def json_default(obj):
    if hasattr(obj, 'item'):
        return obj.item()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f'Object of type {type(obj).__name__} is not JSON serializable')

def get_required_kcs(problem_id, df):
    row = df[df['ProblemID'] == problem_id]
    if row.empty:
        return []
    row = row.iloc[0]
    return [kc for kc in KC_COLUMNS if pd.notna(row.get(kc)) and row.get(kc) == 1]

def get_problem_info(problem_id, df):
    row = df[df['ProblemID'] == problem_id]
    if row.empty:
        return None, None
    row = row.iloc[0]
    return row['Requirement'], row['AssignmentID']

print(f'Students: {len(STUDENT_IDS)}')
print(f'Clusters: {pd.Series(student_clusters).value_counts().to_dict()}')

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Students: 28
Clusters: {'Average': 10, 'High Performer': 10, 'Struggling': 8}


In [5]:
student_data = {}

for sid in STUDENT_IDS:
    df = best_attempts_df[best_attempts_df['SubjectID'] == sid].copy()

    if 'Attempt' in df.columns:
        df = df.sort_values(['ProblemID', 'Score', 'Attempt'])
    else:
        df = df.sort_values(['ProblemID', 'Score'])
    df = df.drop_duplicates(subset=['ProblemID'], keep='last')

    if 'ServerTimestamp' in df.columns:
        df = df.sort_values('ServerTimestamp')
    elif 'Order' in df.columns:
        df = df.sort_values('Order')

    df = df.reset_index(drop=True)
    student_data[sid] = df

total_problems = sum(len(df) for df in student_data.values())
total_nonperfect = sum(len(df[df['Score'] < 1.0]) for df in student_data.values())
print(f'Total problems: {total_problems}')
print(f'Non-perfect (need API calls): {total_nonperfect}')
print(f'Estimated time: {total_nonperfect * SLEEP_SECONDS / 60:.0f} minutes')
print(f'Estimated cost: ${total_nonperfect * 0.0016:.2f}')

Total problems: 1026
Non-perfect (need API calls): 250
Estimated time: 29 minutes
Estimated cost: $0.40


In [6]:
CHECKPOINT_PATH = OUTPUT_DIR / 'exp16_checkpoint_30students.json'

if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, 'r') as f:
        checkpoint = json.load(f)
    completed = set(checkpoint.get('completed', []))
    all_annotations = checkpoint.get('annotations', {})
    print(f'Resuming from checkpoint: {len(completed)} students already done')
else:
    completed = set()
    all_annotations = {}

for sid in STUDENT_IDS:
    sid_str = str(sid)

    if sid_str in completed:
        print(f'Skipping student {sid} (already done)')
        continue

    df = student_data[sid]
    student_annotations = {}
    student_raw = {}

    print(f'\n=== Student {sid} ({student_clusters.get(sid, "?")}), {len(df)} problems ===')

    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        problem_id = int(row['ProblemID'])
        score = float(row['Score'])
        code = str(row.get('Code', '')) if pd.notna(row.get('Code', '')) else ''

        requirement, assignment_id = get_problem_info(problem_id, problem_prompts_df)
        required_kcs = get_required_kcs(problem_id, problem_prompts_df)

        gaps = []
        reasoning = ''
        raw_text = ''
        elapsed = 0
        parse_status = 'skipped'

        if score >= 1.0:
            reasoning = 'Perfect score'
        elif len([l for l in code.split('\n') if l.strip() and not l.strip().startswith('//')]) <= 3 \
             and not any(kw in code.lower() for kw in ['if', 'for', 'while', 'else']):
            reasoning = 'Placeholder code'
        else:
            prompt = build_v3_prompt(
                problem_id=problem_id,
                requirement=requirement,
                assignment_id=assignment_id,
                required_kcs=required_kcs,
                student_code=code,
                score=score,
            )

            try:
                start_time = time.time()
                response = client.models.generate_content(
                    model=MODEL_ID,
                    contents=format_submissions([row.to_dict()]),
                    config=types.GenerateContentConfig(
                        system_instruction=prompt,
                        temperature=0.3,
                        response_mime_type='application/json',
                    ),
                )
                elapsed = round(time.time() - start_time, 3)
                raw_text = response.text if response and response.text else '{}'
                parsed = json.loads(clean_json_response(raw_text))
                reasoning = parsed.get('reasoning', '')
                gaps = parsed.get('knowledge_gaps', [])
                parse_status = 'ok'
            except Exception as exc:
                reasoning = f'ERROR: {exc}'
                gaps = []
                parse_status = 'error'

            time.sleep(SLEEP_SECONDS)

        student_annotations[str(problem_id)] = {'gaps': gaps}
        student_raw[str(problem_id)] = {
            'score': score,
            'required_kcs': required_kcs,
            'assignment_id': int(assignment_id) if assignment_id is not None else 0,
            'reasoning': reasoning,
            'time_sec': elapsed,
            'parse_status': parse_status,
        }

        gap_str = gaps if gaps else '—'
        print(f'  {idx:>3}/{len(df)} P{problem_id} score={score:.3f} gaps={gap_str}')

    student_payload = {
        'rater': 'LLM_Gemini_V3',
        'student_id': sid_str,
        'model_id': MODEL_ID,
        'cluster': student_clusters.get(sid, 'Unknown'),
        'annotations': student_annotations,
        'raw_responses': student_raw,
    }
    per_student_path = OUTPUT_DIR / f'llm_v3_lc_{sid}.json'
    with open(per_student_path, 'w') as f:
        json.dump(student_payload, f, indent=2, default=json_default)

    completed.add(sid_str)
    all_annotations[sid_str] = student_annotations
    checkpoint_data = {
        'completed': list(completed),
        'annotations': all_annotations,
    }
    with open(CHECKPOINT_PATH, 'w') as f:
        json.dump(checkpoint_data, f, default=json_default)

    print(f'  Saved. Checkpoint updated ({len(completed)}/{len(STUDENT_IDS)} students)')

print(f'\nAll {len(STUDENT_IDS)} students complete.')


=== Student 106 (Average), 20 problems ===
    1/20 P13 score=1.000 gaps=—
    2/20 P232 score=1.000 gaps=—
    3/20 P235 score=1.000 gaps=—
    4/20 P234 score=1.000 gaps=—
    5/20 P236 score=1.000 gaps=—
    6/20 P5 score=1.000 gaps=—
    7/20 P233 score=1.000 gaps=—
    8/20 P1 score=1.000 gaps=—
    9/20 P3 score=1.000 gaps=—
   10/20 P12 score=1.000 gaps=—
   11/20 P24 score=1.000 gaps=—
   12/20 P100 score=1.000 gaps=—
   13/20 P101 score=1.000 gaps=—
   14/20 P102 score=1.000 gaps=—
   15/20 P25 score=0.810 gaps=['LogicCompareNum', 'Math+-*/', 'LogicAndNotOr']
   16/20 P28 score=1.000 gaps=—
   17/20 P21 score=1.000 gaps=—
   18/20 P20 score=1.000 gaps=—
   19/20 P17 score=1.000 gaps=—
   20/20 P22 score=1.000 gaps=—
  Saved. Checkpoint updated (1/28 students)

=== Student 9948 (Struggling), 39 problems ===
    1/39 P21 score=1.000 gaps=—
    2/39 P100 score=0.000 gaps=—
    3/39 P101 score=0.960 gaps=['LogicCompareNum']
    4/39 P25 score=1.000 gaps=—
    5/39 P20 score=1.000

In [7]:
all_predictions = []
student_summaries = []

STRUGGLE_THRESHOLD = 1.0

for sid in STUDENT_IDS:
    sid_str = str(sid)
    filepath = OUTPUT_DIR / f'llm_v3_lc_{sid}.json'

    with open(filepath, 'r') as f:
        data = json.load(f)

    annotations = data['annotations']
    raw = data['raw_responses']
    cluster = data.get('cluster', student_clusters.get(sid, 'Unknown'))

    problems = []
    for pid, info in raw.items():
        problems.append({
            'pid': pid,
            'score': info['score'],
            'required_kcs': info['required_kcs'],
            'assignment_id': info['assignment_id'],
            'gaps': annotations.get(pid, {}).get('gaps', []),
        })

    problems.sort(key=lambda x: (x['assignment_id'], int(x['pid'])))

    student_preds = []

    for i, current in enumerate(problems):
        if not current['gaps']:
            continue

        for gap_kc in current['gaps']:
            found = False
            for j in range(i + 1, len(problems)):
                future = problems[j]
                if gap_kc in future['required_kcs']:
                    still_struggling = future['score'] < STRUGGLE_THRESHOLD
                    pred = {
                        'SubjectID': sid,
                        'Cluster': cluster,
                        'GapKC': gap_kc,
                        'SourceProblem': current['pid'],
                        'SourceScore': current['score'],
                        'SourceOrder': i,
                        'FutureProblem': future['pid'],
                        'FutureScore': future['score'],
                        'FutureOrder': j,
                        'StillStruggling': still_struggling,
                        'Prediction': 'TP' if still_struggling else 'FP',
                    }
                    student_preds.append(pred)
                    all_predictions.append(pred)
                    found = True
                    break

            if not found:
                all_predictions.append({
                    'SubjectID': sid, 'Cluster': cluster, 'GapKC': gap_kc,
                    'SourceProblem': current['pid'], 'SourceScore': current['score'],
                    'SourceOrder': i, 'FutureProblem': None, 'FutureScore': None,
                    'FutureOrder': None, 'StillStruggling': None, 'Prediction': 'NO_FUTURE',
                })

    validatable = [p for p in student_preds if p['Prediction'] in ('TP', 'FP')]
    if validatable:
        tp = sum(1 for p in validatable if p['Prediction'] == 'TP')
        hit = tp / len(validatable)
        student_summaries.append({
            'SubjectID': sid, 'Cluster': cluster,
            'Validatable': len(validatable),
            'TP': tp, 'FP': len(validatable) - tp,
            'HitRate': round(hit, 3),
        })

pred_df = pd.DataFrame(all_predictions)
validatable_df = pred_df[pred_df['Prediction'].isin(['TP', 'FP'])].copy()

tp = len(validatable_df[validatable_df['Prediction'] == 'TP'])
fp = len(validatable_df[validatable_df['Prediction'] == 'FP'])
hit_rate = tp / len(validatable_df) if len(validatable_df) > 0 else 0

print(f'=== V3 Predictive Validation (30 Students) ===')
print(f'Validatable predictions: {len(validatable_df)}')
print(f'True Positives: {tp}')
print(f'False Positives: {fp}')
print(f'Hit Rate: {hit_rate:.3f} ({hit_rate*100:.1f}%)')

print(f'\nPer-Cluster Hit Rate:')
summary_df = pd.DataFrame(student_summaries)
cluster_agg = summary_df.groupby('Cluster').agg(
    Students=('SubjectID', 'count'),
    TotalValidatable=('Validatable', 'sum'),
    TotalTP=('TP', 'sum'),
    TotalFP=('FP', 'sum'),
).reset_index()
cluster_agg['HitRate'] = (cluster_agg['TotalTP'] / cluster_agg['TotalValidatable']).round(3)
display(cluster_agg)

print(f'\nPer-KC Hit Rate:')
print(f"{'KC':<20} {'TP':>4} {'FP':>4} {'Total':>6} {'Hit%':>7}")
print('-' * 45)
for kc in KC_COLUMNS:
    kc_data = validatable_df[validatable_df['GapKC'] == kc]
    if len(kc_data) > 0:
        kc_tp = len(kc_data[kc_data['Prediction'] == 'TP'])
        kc_hit = kc_tp / len(kc_data)
        print(f'{kc:<20} {kc_tp:>4} {len(kc_data)-kc_tp:>4} {len(kc_data):>6} {kc_hit:>6.1%}')

print(f'\nPer-Student Summary:')
display(summary_df.sort_values(['Cluster', 'HitRate'], ascending=[True, False]))

=== V3 Predictive Validation (30 Students) ===
Validatable predictions: 226
True Positives: 132
False Positives: 94
Hit Rate: 0.584 (58.4%)

Per-Cluster Hit Rate:


,Cluster,Students,TotalValidatable,TotalTP,TotalFP,HitRate
0,Average,6,68,18,50,0.265
1,High Performer,4,10,0,10,0.000
2,Struggling,6,148,114,34,0.770



Per-KC Hit Rate:
KC                     TP   FP  Total    Hit%
---------------------------------------------
If/Else                22   13     35  62.9%
While                   2    0      2 100.0%
For                     6    8     14  42.9%
NestedFor               0    1      1   0.0%
Math+-*/                9    7     16  56.2%
Math%                   1    3      4  25.0%
LogicAndNotOr          17   15     32  53.1%
LogicCompareNum        27   24     51  52.9%
LogicBoolean            1    1      2  50.0%
StringFormat            8    1      9  88.9%
StringConcat            4    3      7  57.1%
StringIndex            13    8     21  61.9%
StringLen               4    2      6  66.7%
StringEqual             3    2      5  60.0%
CharEqual               3    2      5  60.0%
ArrayIndex              8    4     12  66.7%
DefFunction             4    0      4 100.0%

Per-Student Summary:


,SubjectID,Cluster,Validatable,TP,FP,HitRate
13,14359,Average,17,7,10,0.412
6,14186,Average,8,3,5,0.375
2,10083,Average,14,4,10,0.286
4,10224,Average,9,2,7,0.222
11,14316,Average,17,2,15,0.118
0,106,Average,3,0,3,0.000
5,13365,High Performer,7,0,7,0.000
8,14205,High Performer,1,0,1,0.000
9,14289,High Performer,1,0,1,0.000
10,14296,High Performer,1,0,1,0.000


In [8]:
random.seed(42)
NUM_TRIALS = 1000

total_slots = 0
total_flagged = 0
all_problems_by_student = {}

for sid in STUDENT_IDS:
    filepath = OUTPUT_DIR / f'llm_v3_lc_{sid}.json'
    with open(filepath, 'r') as f:
        data = json.load(f)

    problems = []
    for pid, info in data['raw_responses'].items():
        gaps = data['annotations'].get(pid, {}).get('gaps', [])
        problems.append({
            'pid': pid, 'score': info['score'],
            'required_kcs': info['required_kcs'],
            'assignment_id': info['assignment_id'],
        })
        total_slots += len(info['required_kcs'])
        total_flagged += len(gaps)

    problems.sort(key=lambda x: (x['assignment_id'], int(x['pid'])))
    all_problems_by_student[sid] = problems

gap_rate = total_flagged / total_slots if total_slots > 0 else 0
print(f'V3 gap rate: {gap_rate:.3f} ({total_flagged}/{total_slots})')

random_hit_rates = []
for trial in range(NUM_TRIALS):
    rand_preds = []
    for sid, problems in all_problems_by_student.items():
        for i, current in enumerate(problems):
            rand_gaps = [kc for kc in current['required_kcs'] if random.random() < gap_rate]
            for gap_kc in rand_gaps:
                for j in range(i + 1, len(problems)):
                    future = problems[j]
                    if gap_kc in future['required_kcs']:
                        rand_preds.append(future['score'] < STRUGGLE_THRESHOLD)
                        break
    if rand_preds:
        random_hit_rates.append(sum(rand_preds) / len(rand_preds))

rand_mean = statistics.mean(random_hit_rates) if random_hit_rates else 0.0
rand_std = statistics.pstdev(random_hit_rates) if len(random_hit_rates) > 1 else 0.0

print(f'\n=== V3 vs Random Baseline (30 Students) ===')
print(f'V3 hit rate:      {hit_rate:.3f}')
print(f'Random baseline:  {rand_mean:.3f} ± {rand_std:.3f}')
print(f'Improvement:      {hit_rate - rand_mean:+.3f}')
print(f'V3 > random (2σ): {hit_rate > rand_mean + 2 * rand_std}')
print(f'V3 > random (3σ): {hit_rate > rand_mean + 3 * rand_std}')

summary = {
    'experiment': 'Exp16 - Learning Curve Validation (30 Students)',
    'students': STUDENT_IDS,
    'num_students': len(STUDENT_IDS),
    'struggle_threshold': STRUGGLE_THRESHOLD,
    'total_problems': sum(len(p) for p in all_problems_by_student.values()),
    'validatable': len(validatable_df),
    'true_positives': int(tp),
    'false_positives': int(fp),
    'v3_hit_rate': round(hit_rate, 4),
    'v3_gap_rate': round(gap_rate, 4),
    'random_mean': round(rand_mean, 4),
    'random_std': round(rand_std, 4),
    'v3_beats_random_2sigma': bool(hit_rate > rand_mean + 2 * rand_std),
    'v3_beats_random_3sigma': bool(hit_rate > rand_mean + 3 * rand_std),
    'per_cluster': cluster_agg.to_dict('records'),
    'student_summaries': student_summaries,
}

with open(OUTPUT_DIR / 'exp16_learning_curve_30students.json', 'w') as f:
    json.dump(summary, f, indent=2, default=json_default)
pred_df.to_csv(OUTPUT_DIR / 'exp16_predictions_30students.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'exp16_per_student_summary_30students.csv', index=False)

print(f'\nSaved all results to {OUTPUT_DIR}')

V3 gap rate: 0.050 (265/5306)

=== V3 vs Random Baseline (30 Students) ===
V3 hit rate:      0.584
Random baseline:  0.264 ± 0.028
Improvement:      +0.320
V3 > random (2σ): True
V3 > random (3σ): True

Saved all results to results/learning_curve
